# comprendre le code petit à petit

preparation


In [1]:
using  Pkg

cd(@__DIR__)
Pkg.activate("../..")
ParamFile = "../test/testparam.csv"
include("../src/DSM1D.jl")
include("../src/batchUseful.jl")
using .DSM1D
using DIVAnd,CairoMakie
using Interpolations
#import GLMakie
import CairoMakie
using Colors
include("../src/batchStagYY.jl")
include("../src_Neurthino/Neurthino.jl")
using .Neurthino
include("../src_Neurthino/usefulFunctionsToPlot.jl")
include("../src_Neurthino/NeurthinoRelated.jl")
include("../src_Neurthino/premFunctions.jl")
boolFlat = true # 2d

  Activating 

  1.945726 seconds (4.56 M allocations: 208.356 MiB, 2.22% gc time, 99.93% compilation time)


project at `~/Documents/Github/flexibleDSM`


true

# geometry

In [2]:
detectorPosition = Float32[1.8340824f6, 7.7562185f6] #you have to put manually the position of the detector to compare (in X and Y)
zposition = 2.5e3 # depth of dectecor in metre

2500.0

# import geodynamic models -> need to change

In [3]:
#small patch for my environment
dir="/Users/nobuaki/Documents/MantleConvectionTakashi/op_old_full_mars_2025/"
#dir="C:/documents/github/data/op_old_full_mars_2025"

iTime=200
#iTime = 3

200

In [4]:
rhoFiles=myListDir(dir; pattern=r"test_rho\d");
compositionFiles=myListDir(dir; pattern=r"test_c\d");
temperatureFiles=myListDir(dir; pattern=r"test_t\d");
wtrFiles=myListDir(dir; pattern=r"test_wtr\d")


201-element Vector{Any}:
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00000"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00001"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00002"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00003"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00004"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00005"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00006"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00007"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00008"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00009"
 ⋮
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//test_wtr00192"
 "/Users/nobuaki/Documents/Mantle" ⋯ 23 bytes ⋯ "d_full_mars_2025//tes

# some inputs needed

In [5]:

n_pts = 100 # nombre de segments qui divisent chaque rayon
n_vectors = 100 # nb de rayons
n_energies = 100 # nb of energy bins


100

# new functions added 

In [6]:


function readStagYYFilesAverage(file)
    magic, inputILEN, byte_reverse_in, io=read_magic(file)
    magic,nval = evaluate_nval_from_magicNumber(magic)
    intDataType,floatDataType,dx,dy,nx,ny,nz,nb,nnx,nny,nnz,nnb,rcmb,iStep,time,zc,boolSpherical=read_header(io,inputILEN,magic)
   
    nodes=nothing
    boolFlat = true

    Xnode=[]
    Ynode=[]    
    Znode=[]


   
    if nx == 1
       boolFlat = true
    end
 


    if boolSpherical
        # these are the grid points including the boundaries
        r = rcmb .+ zc[1,1:nz+1]
        
        theta = (collect(1:1:nx+1) .- (nx+1)) .* dx .+ 0.5*π
        phi = (collect(1:1:ny+1) .- (ny+1)) .* dy
        

        # these are the midpoints 
        r_mid = rcmb .+ zc[2,1:nz]
        theta_mid = (collect(1:1:nx) .- (nx+1)) .* dx .+ 0.5*π .+ 0.5*dx
        phi_mid = phi = (collect(1:1:ny) .- (ny+1)) .* dy .+ 0.5*dy
        
        # interpolation should be done with the midpoints + end points

        r_new = prepend!(r_mid,r[1])
        r_new = push!(r_mid,r[end])
        minR = r[1]
        maxR = r[end]
        rC=r_new

        theta_new = prepend!(theta_mid,theta[1])
        theta_new = push!(theta_new,theta[end])
        minθ = theta[1]
        maxθ = theta[end]
        phi_new = prepend!(phi_mid,phi[1])
        phi_new = push!(phi_new,phi[end])
        minϕ = phi[1]
        maxϕ = phi[end]
        if boolFlat
            minθ = 0.
            maxθ = 0.
            theta_new = (π/2)  # if flat, we are looking at the 
            nodes=(theta_new,phi_new,r_new)
        else   
            nodes=(theta_new,phi_new,r_new)
        end

    
    else
        # do not know if this works (certainly not! try to mimic the code above for the spherical case)
        x=collect(1:1:nx+1) .* dx
        y=collect(1:1:ny+1) .* dy
        z=zc
        nodes=(x,y,z)
    end

    

    theta_new_piCoef = theta_new ./ π
    phi_new_piCoef = phi_new ./ π

    if boolSpherical
        for rTemp in r_new
            for phiTemp in phi_new_piCoef
                cosPhiPi = cospi(phiTemp)
                sinPhiPi = sinpi(phiTemp)
               
                for thetaTemp in theta_new_piCoef
                    cosThetaPi = cospi(thetaTemp)
                    sinThetaPi = sinpi(thetaTemp) 
                    Xnode = push!(Xnode,rTemp*sinThetaPi*cosPhiPi)
                    Ynode = push!(Ynode,rTemp*sinThetaPi*sinPhiPi)
                    Znode = push!(Znode,rTemp*cosThetaPi)

                end
            end
        end
    else
        @error "cartesian format: not yet developed"
    end

   #coord[1,ix,iy,iz] = r[iz]*sin(theta)*cos(phi)
   #coord[2,ix,iy,iz] = r[iz]*sin(theta)*sin(phi)
   #coord[3,ix,iy,iz] = r[iz]*cos(theta)
   
   
   #field = binRead(io,floatDataType,(nx)*(ny)*(nz))
   #field = binRead(io,Float64,nx*ny*nz)
   #rawField =field
   #newField = nothing

   rawField = readField(io,floatDataType,nval,nx,ny,nz,nb,nnx,nny,nnz,nnb)

   
    if boolFlat
        #field=reshape(field, (ny,nz)) 

        field = zeros(floatDataType,ny,nz)
        field[1:ny,1:nz]=rawField[1,1,1:ny,1:nz,1]


        newField = zeros(floatDataType,ny+2,nz+2)

        # the interior

        newField[2:ny+1,2:nz+1] = field[1:end,1:end]

        # 4 'surfaces'

        newField[1,2:nz+1] = field[1,1:nz]
        newField[end,2:nz+1] = field[end,1:nz]
        newField[2:ny+1,1] = field[1:ny,1]
        newField[2:ny+1,end] = field[1:ny,end]

        # 4 'endpoints'

        newField[1,1,1]=field[1,1,1]
        newField[1,end,1]=field[1,end,1]
        newField[1,1,end]=field[1,1,end]
        newField[1,end,end]=field[1,end,end]


    else
        #field=reshape(field, (nx,ny,nz)) 
        field = zeros(floatDataType,nx,ny,nz)
        field[1:ny,1:nz]=rawField[1,1:nx,1:ny,1:nz,1]

        newField = zeros(floatDataType,nx+2,ny+2,nz+2)

        # the interior

        newField[2:nx+1,2:ny+1,2:nz+1] = field[1:nx,1:ny,1:nz]

        # 6 'surfaces'
        
        newField[1,2:ny+1,2:nz+1] = field[1,1:ny,1:nz]
        newField[end,2:ny+1,2:nz+1] = field[end,1:ny,1:nz]
        newField[2:nx+1,1,2:nz+1] = field[1:nx,1,1:nz]
        newField[2:nx+1,end,2:nz+1] = field[1:nx,end,1:nz]
        newField[2:nx+1,2:ny+1,1] = field[1:nx,1:ny,1]
        newField[2:nx+1,2:ny+1,end] = field[1:nx,1:ny,end]

        # 8 'endpoints'
        newField[1,1,1]=field[1,1,1]
        newField[end,1,1]=field[end,1,1]
        newField[1,end,1]=field[1,end,1]
        newField[end,end,1]=field[end,end,1]
        newField[1,1,end]=field[1,1,end]
        newField[end,1,end]=field[end,1,end]
        newField[1,end,end]=field[1,end,end]
        newField[end,end,end]=field[end,end,end]

    end

    if boolFlat
        avNewField = similar(newField)
        diffNewField = similar(newField)
        for iz in 1:nz+2
            average = sum(newField[:,iz])
            average /= Float64(ny+2)
            avNewField[:,iz] .= average
        end
        #@show size(avNewField)
        diffNewField = newField .- avNewField
        #Création de diffpfield 
        newField=reshape(newField,(ny+2)*(nz+2))
        avNewField=reshape(avNewField,(ny+2)*(nz+2))
        diffNewField=reshape(diffNewField,(ny+2)*(nz+2))
        return newField, avNewField, diffNewField, Xnode, Ynode, rcmb
    else
        newField=reshape(newField,(nx+2)*(ny+2)*(nz+2))
        return newField, Xnode, Ynode, Znode, rcmb
    end
   #fieldInterpolated=interpolate(nodes,newField,Gridded(Linear()))

end



function ZOverAwithWaterWithThreeLayeredZA1Dmodel(wtrfieldCartesien,xCartesien,yCartesien)#avec le tableau de Lucas (p.4 Unveiling the Outer core with not)
    ZoverAwtr=similar(wtrfieldCartesien)
    ZoverAwtr.= 0.0
    ZoverA=similar(wtrfieldCartesien)
    ZoverA.= 0.0
    for i in eachindex(xCartesien), j in eachindex(yCartesien)
        x=xCartesien[i]
        y=yCartesien[j]
        r=sqrt(x*x+y*y)
        if 0.0 ≤ r < 1221000.5
            ZoverA[i,j]=0.466
        elseif  1221000.5 <= r < 3480000.0
            ZoverA[i,j]=0.466
        elseif 3480000.0 <= r < 6368000.0
            ZoverA[i,j]=0.496
        end
        #if 0.0 ≤ r < 6368000.0 #a rajouter si on veut que l'espace ai un Z/A de 0
            ZoverAwtr[i,j] = wtrfieldCartesien[i,j]*(5.0/9.0)+ (1.0-wtrfieldCartesien[i,j])*ZoverA[i,j]
        #end
     end

    
    return ZoverAwtr,ZoverA    
end

ZOverAwithWaterWithThreeLayeredZA1Dmodel (generic function with 1 method)

# functions used and modified for neutrino oscillation computation

In [7]:

function lineDensityElectron2D(n_pts, effectiveρModel, positionDetector, NeutrinoSource, colorname, ax1, dR)
    #draw a line between positionDetector and NeutrinoSource (coordinates) and give the density/distance profile
    #dependencies : Makie

    #fig, ax, fi = myPREMPlot2DConvectionModel(iTime, "rho", rhoFiles)
    fi = effectiveρModel

    x_phys = range(positionDetector[1], NeutrinoSource[1], length=n_pts)
    y_phys = range(positionDetector[2], NeutrinoSource[2], length=n_pts)  
    
    lines!(ax, x_phys,y_phys, color=colorname)  # (x,y)_phys in m
    display(fig)

    x_grid = x_phys ./dR
    y_grid = y_phys ./dR
    itp = interpolate(fi, BSpline(Linear()), OnGrid())

    densGrids = Float64[]
    for i in eachindex(x_grid)
        x = x_grid[i]
        y = y_grid[i]
        push!(densGrids, itp(x,y)*1e-3) #g/cm3
    end

    dens=Float64[]
    for i in eachindex(densGrids)[1:end-1]
        push!(dens, 0.5*(densGrids[i]+densGrids[i+1]))
    end

    segmentLengthInKm = sqrt((x_phys[2]-x_phys[1])^2 + (y_phys[2]-y_phys[1])^2) * 1.e-3
    sections = segmentLengthInKm .* ones(Float64,n_pts-1) 
    dist = segmentLengthInKm*collect(0:1:n_pts-1) #km

    lines!(ax1, dist, densGrids, color=colorname)
    return dens, sections
end


lineDensityElectron2D (generic function with 1 method)

In [8]:
function creationPaths(n_vectors, pos,zposition,effectiveρModel;center = [6.5e6, 6.5e6])

    densities_list, sections_list = vectorsFromDetector(n_vectors, pos,zposition,effectiveρModel; center=center) 
    paths = Vector{Path}(undef, n_vectors)  

    for i in eachindex(paths)
        paths[i]= Path(densities_list[i],sections_list[i])
    end

    return paths
end

creationPaths (generic function with 1 method)

In [9]:
function correctedPosition(x,y, zposition; center = [6.5e6, 6.5e6], earth_radius = 6.371e6)
    #to place the detector precisely on the surface, then the detector is buried for zposition

    dx = x - center[1]
    dy = y - center[2]

    real_pos = earth_radius - zposition #m
    dist_radiale = sqrt(dx^2 + dy^2)
    new_x = center[1] + real_pos*dx/dist_radiale
    new_y = center[2] + real_pos*dy/dist_radiale
    return new_x, new_y, zposition
end

correctedPosition (generic function with 1 method)

In [10]:

function sourcePosition(center, positionDetector, n_vectors, zposition; earthRadius = 6.371e6)
    #to get the position of the different sources

    (xc, yc) = center[1], center[2] #m 
    (xd, yd) = positionDetector[1], positionDetector[2] #m
    XY = []

    cos_θ = range(-1, 0, length = n_vectors)
    θ = posOrNeg(cos_θ, :positive)
    cos_epi = cos.(2 .*θ .- π)
    sin_epi = sin.(2 .*θ .- π)
    rotation = cos_epi .+ im .* sin_epi


    for i in eachindex(cos_θ)
        equ = ((xd - xc) + (yd-yc)*im) * rotation[i]
        X = real(equ)+xc
        Y = imag(equ)+yc
        # this is the position with zposition below

        newX = nothing
        newY = nothing
        if cos_θ[i] !== 0.0
            if X-xd != 0.0
                slope = (Y-yd)/(X-xd)

                a = 1+slope^2
                b = -2*xc + 2*Y*slope-2*slope^2*X-2*slope*yc
                c = xc^2 + Y^2 - 2*Y*slope*X + slope^2*X^2 -2*Y*yc + 2*slope*X*yc +yc^2 - earthRadius^2
                sol1,sol2 = solveQuadraticEquation(a,b,c)

                if (sol1-xd)*(X-xd)>0.0
                    newX = sol1
                    newY = Y + slope* (newX - X)
                else
                    newX = sol2
                    newY = Y + slope* (newX - X)
                end

            else
                slope = (X-xd)/(Y-yd)
                
                a = 1+slope^2
                b = -2*yc + 2*X*slope-2*slope^2*Y-2*slope*xc
                c = yc^2 + X^2 - 2*X*slope*Y + slope^2*Y^2 -2*X*xc + 2*slope*Y*xc +xc^2 - earthRadius^2
                sol1,sol2 = solveQuadraticEquation(a,b,c)
                
                if (sol1-yd)*(Y-yd)>0.0
                    newY = sol1
                    newX = Y + slope* (newY - Y)
                else
                    newY = sol2
                    newX = Y + slope* (newY - Y)
                end

            end

        else
            segmentfromDtoS = sqrt(earthRadius^2-(earthRadius-zposition)^2)
            if θ[i] > 0
                newX = xd - (yd-yc)/(earthRadius-zposition)*segmentfromDtoS
                newY = yd + (xd-xc)/(earthRadius-zposition)*segmentfromDtoS
            else 
                newX = xd + (yd-yc)/(earthRadius-zposition)*segmentfromDtoS
                newY = yd - (xd-xc)/(earthRadius-zposition)*segmentfromDtoS
            end
        end
        push!(XY, (newX,newY))

    end
    return XY

end

sourcePosition (generic function with 1 method)

In [11]:

function vectorsFromDetector(n_vectors, pos, zposition,effectiveρModel ;center = [6.5e6, 6.5e6])
    #draw n_vectors (diff θ) from a detector (placed by interaction) and return density profiles for each vector through the Earth
    #dependencies : GLMakie, Makie, Colors


    
    #@show pos
    x, y = pos[1], pos[2]
    new_x, new_y,zposition = correctedPosition(x,y, zposition) 
    XY = sourcePosition((center[1], center[2]), (new_x, new_y), n_vectors, zposition)

    segments_pts = []
    for source in XY
        push!(segments_pts, (new_x, new_y))
        push!(segments_pts, (source[1], source[2]))
    end

    
    CairoMakie.activate!()
    fig1 = Figure()
    ax1 = Axis(fig1[1,1], xlabel="Path (km)", ylabel="Density (g/cm3)")


    densities_list = []
    sections_list = []
    for i in eachindex(XY)
        colorname = rand(collect(keys(Colors.color_names)))
        detector = new_x, new_y
        source = XY[i][1], XY[i][2]
        dens, section = lineDensityElectron2D(n_pts, effectiveρModel, detector,source, colorname, ax1, dR)

        push!(densities_list, dens)
        push!(sections_list, section)

    end

    display(fig1)
    return densities_list, sections_list
end


vectorsFromDetector (generic function with 2 methods)

# the computation starts here

## making masks for interpolation

In [12]:
# Cartesian grids and interpolation
correlationLength=(20e3,20e3,20e2) # not yet fully understood this for DIV interpolation
epsilon2 =1.;

minX,maxX,nX = -6500e3, 6500e3, 521
minY,maxY,nY = minX,maxX,nX
dR = (maxX-minX)/(nX-1) # pas

25000.0

In [13]:

if boolFlat
    #nZ=1
    minZ=0.0
    maxZ=0.0
    tmpX=correlationLength[1]
    tmpY=correlationLength[2]
    correlationLength=(tmpX,tmpY)

    mask,(pm,pn),(xi,yi) = DIVAnd_rectdom(range(minX,stop=maxX,length=nX),
                                            range(minY,stop=maxY,length=nY));
else

    mask,(pm,pn,po),(xi,yi,zi) = DIVAnd_rectdom(range(minX,stop=maxX,length=nX),
                                            range(minY,stop=maxY,length=nY),
                                            range(minZ,stop=maxZ,length=nZ));
end



(Bool[1 1 … 1 1; 1 1 … 1 1; … ; 1 1 … 1 1; 1 1 … 1 1], ([4.0e-5 4.0e-5 … 4.0e-5 4.0e-5; 4.0e-5 4.0e-5 … 4.0e-5 4.0e-5; … ; 4.0e-5 4.0e-5 … 4.0e-5 4.0e-5; 4.0e-5 4.0e-5 … 4.0e-5 4.0e-5], [4.0e-5 4.0e-5 … 4.0e-5 4.0e-5; 4.0e-5 4.0e-5 … 4.0e-5 4.0e-5; … ; 4.0e-5 4.0e-5 … 4.0e-5 4.0e-5; 4.0e-5 4.0e-5 … 4.0e-5 4.0e-5]), ([-6.5e6 -6.5e6 … -6.5e6 -6.5e6; -6.475e6 -6.475e6 … -6.475e6 -6.475e6; … ; 6.475e6 6.475e6 … 6.475e6 6.475e6; 6.5e6 6.5e6 … 6.5e6 6.5e6], [-6.5e6 -6.475e6 … 6.475e6 6.5e6; -6.5e6 -6.475e6 … 6.475e6 6.5e6; … ; -6.5e6 -6.475e6 … 6.475e6 6.5e6; -6.5e6 -6.475e6 … 6.475e6 6.5e6]))

In [ ]:
file = rhoFiles[iTime]
ρfield, avρField, diffρField, Xnode, Ynode, rcmb = readStagYYFilesAverage(file)
XXnode=Xnode
YYnode=Ynode
extendToCoreWithρ!(ρfield, XXnode, YYnode, rcmb, dR, iCheckCoreModel=false)
#extendToCoreWithρ!(avρField, Xnode, Ynode, rcmb, dR, iCheckCoreModel=false)
#extendToCoreWithρ!(diffρField, Xnode, Ynode, rcmb, dR, iCheckCoreModel=false)
file2 = wtrFiles[iTime]
wtrfield, Xnode, Ynode, rcmb = readStagYYFiles(file2)
#quarterDiskExtrapolationRawGrid!(field, Xnode, Ynode)
fi,_ = DIVAndrun(mask,(pm,pn),(xi,yi),(Xnode,Ynode),diffρField,correlationLength,epsilon2);

ρfieldCartesien,_ = DIVAndrun(mask,(pm,pn),(xi,yi),(XXnode,YYnode),ρfield,correlationLength,epsilon2);
#la fonctioin divandrun interpole en cartésien car les données initiales sont en polaires

diam = maxX - minX
x = range(0, diam, length=size(fi)[1])
y = range(0, diam, length=size(fi)[2])

fig = Figure()
ax = Axis(fig[1,1], aspect = 1)
colormap = myChoiceColormap(fieldname)
hm=heatmap!(ax,x, y, fi, colormap=colormap)
Colorbar(fig[:,2], hm, label="Density perturbation to the  (kg/m3)")
display(fig)

In [ ]:
wtrfieldCartesien,_=DIVAndrun(mask,(pm,pn),(xi,yi),(Xnode,Ynode),wtrfield,correlationLength,epsilon2)
diam = maxX - minX
x = range(0, diam, length=size(wtrfieldCartesien)[1])
y = range(0, diam, length=size(wtrfieldCartesien)[2])

fig = Figure()
ax = Axis(fig[1,1], aspect = 1)
colormap = myChoiceColormap(fieldname)
hm=heatmap!(ax,x, y,wtrfieldCartesien, colormap=colormap)
Colorbar(fig[:,2], hm, label=L"H_2O\ \mathrm{[wt\%]}")
display(fig)


AU DESSUS : DIFFERENCE ENTRE LE MODELE DE DENSITÉ 'oignon' ET CELUI SUR LES GRILLES D'INTERPOLATION CARTHESIENNE


POUR RECUPERER Ne (densité d'e-): multiplier ρfield par Z/A correspondant à chaque couche (cf papier de Lucas Maderer)

PLOTTER UN FIELD DE Z/A en cartésien, à partir des données du papier de lucas maderer


In [ ]:
xCartesien = minX .+ dR*collect(1:1:nX) .- dR;
yCartesien = minY .+ dR*collect(1:1:nY) .- dR;

## first we generate Ne based on the radius and water distribution

In [ ]:
@show maximum(ZOverAwtr)
@show minimum(x for x in ZOverAwtr if x > 0.3)

In [ ]:

ZOverAwtr, ZoverA=ZOverAwithWaterWithThreeLayeredZA1Dmodel(wtrfieldCartesien,xCartesien,yCartesien)
diam = maxX - minX
x = range(0, diam, length=size(ZOverAwtr)[1])
y = range(0, diam, length=size(ZOverAwtr)[2])

fig = Figure()
ax = Axis(fig[1,1], aspect = 1)
colormap = myChoiceColormap(fieldname)
hm=heatmap!(ax,x, y, ZOverAwtr, colormap=:viridis, colorrange = (0.495,0.498))
Colorbar(fig[:,2], hm, label="Z/A")
display(fig)

In [ ]:
NeFieldCartesien = ZOverAwtr .* ρfieldCartesien;

In [ ]:
fi=NeFieldCartesien
diam = maxX - minX
x = range(0, diam, length=size(fi)[1])
y = range(0, diam, length=size(fi)[2])

fig = Figure()
ax = Axis(fig[1,1], aspect = 1)
colormap = myChoiceColormap(fieldname)
hm=heatmap!(ax,x, y, fi, colormap=colormap)
Colorbar(fig[:,2], hm, label=L"\mathrm{electron\ number\ density} [\mathrm{m}^{-3}]")


display(fig)

As Neurthino.jl assumes Z/A = 0.5, we need to fake effective density from ne


In [ ]:
effectiveρ = NeFieldCartesien ./ 0.5;

## We then need to work on PMNS matrix construction and baseline computation

In [ ]:
osc = OscillationParameters(3)
setθ!(osc, 1=>2, 0.59)
setθ!(osc, 1=>3, 0.15)
setθ!(osc, 2=>3, 0.84)
setδ!(osc, 1=>3, 3.86)
setΔm²!(osc, 2=>3, -2.523e-3)
setΔm²!(osc, 1=>2, -7.39e-5)
U = PMNSMatrix(osc)
H = Hamiltonian(osc)
Uround = roundExt.(U, 0.01)
Hround = roundExt.(H, 0.00001)
cos_θ = range(-1, 0, length = n_vectors);



In [ ]:

paths = creationPaths(n_vectors, positionDetector, zposition,effectiveρ)
energies = 10 .^ range(0, stop=2, length=n_energies)
probs = Pνν(Uround, Hround, energies, paths)[:,:,1,2]
matprobs=parent(probs)

fig = Figure()
ax = Axis(fig[1,1], aspect = 1, xscale=log10, xlabel="Energy (GeV)", ylabel="cos(θ)")
hm=heatmap!(ax, energies, cos_θ, matprobs, colormap=cgrad(:inferno))
Colorbar(fig[:,2], hm, label="Probability")
display(fig)